# Stage 4 — independent defect realizations

Exact finite-`L=30` robustness check for **three fixed seeds** `20261008`, `20261009`, `20261010` with identical lattice and analysis regions; only the hexagon positions change. This regenerates the central test of the preceding Stage-4 realization study: the one-sided zero-mode occupation-exchange mechanism repeats, **but the jump amplitude and even its sign need not repeat**.

This is **not** disorder-averaged convergence and does **not** address dependence on system size or the location of the physical boundary. No parameter selection is conditioned on attractive-looking peaks.

**Cost:** 12 dense diagonalizations of an 1800×1800 complex Hermitian matrix (plus the pristine calculation); allow substantial laptop time and RAM. Change `L` downward for a quick test, but the reference numerical values then cease to apply.

In [ ]:
from pathlib import Path
import sys
REPO = Path.cwd().resolve()
if not (REPO / 'src').exists():
    REPO = REPO.parent
if not (REPO / 'src').exists():
    raise FileNotFoundError('Start Jupyter in the repository root or notebooks/ directory.')
sys.path.insert(0, str(REPO / 'src'))
import numpy as np
import matplotlib.pyplot as plt
from stage4_local_marker import HaldaneOBC, FigureRegions, exact_marker
from stage4_local_marker.analysis import (secular_roots, root_in_window,
    site_masks, region_sums, exact_crossing_sample,
    occupation_swap_jump, frozen_reference_reconstruction)


In [ ]:
L=30
SEEDS=[20261008,20261009,20261010]
EPS=1e-6
model=HaldaneOBC(L,L,t=1.0,t2=.02,m=0.0)
geo=FigureRegions(model,defect_width=10,integration_width=15)
H0=model.pristine();pos=model.positions();mask=geo.integration_mask()
masks=site_masks(geo,edge_layers=2)
base=exact_marker(H0,pos,max_dim=2*L*L).marker
summaries=[];curves={}
for seed in SEEDS:
    defects=geo.choose_defects(3,seed)
    W=model.defect_unit_operator(defects)
    roots=secular_roots(H0,W)
    centers={'left':root_in_window(roots,-1.5,-.95),
             'right':root_in_window(roots,.6,1.0)}
    curves[seed]={}
    for branch,hc in centers.items():
        neg=exact_crossing_sample(H0,W,pos,mask,base,hc-EPS,max_dim=2*L*L)
        posi=exact_crossing_sample(H0,W,pos,mask,base,hc+EPS,max_dim=2*L*L)
        swap=occupation_swap_jump(neg,posi,pos,base,mask)
        contributions=region_sums(posi.delta_marker,masks)
        summaries.append({'seed':seed,'branch':branch,'h_c':hc,
                         'jump':swap['exact_jump'],'swap_error':swap['relative_error'],
                         'B_upper':contributions['B'],
                         'R_minus_B_upper':contributions['R_minus_B'],
                         'edge_upper':contributions['edge']})
        curves[seed][branch]=(hc,neg.integrated_marker,posi.integrated_marker)
    print('Finished seed',seed,'defects:',defects,flush=True)

import pandas as pd
summary=pd.DataFrame(summaries)
display(summary.round(5))

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(11,4),constrained_layout=True)
for a,branch in zip(axs,['left','right']):
    values=summary[summary.branch==branch]
    a.bar(values.seed.astype(str),values.jump)
    a.axhline(0,color='gray',lw=.8)
    a.set(title=f'{branch.capitalize()} zero-energy crossing',
          xlabel='Seed',ylabel='One-sided marker jump in R')
plt.show()
print('Mechanism: state exchange is reproducible, but geometry controls sign and magnitude.')

In [ ]:
SAVE = False
if SAVE:
    out=REPO/'outputs'/'robustness';out.mkdir(parents=True,exist_ok=True)
    summary.to_csv(out/'summary.csv',index=False)
    fig.savefig(out/'one_sided_jumps.png',dpi=180)
    print(out)